# DBSCAN 취약권역 집중지역 확인

- 목적: 취약권역에 포함된 취약격자가 어느 구/행정동에 많이 모여 있는지 확인한다.
- 기준: `DBSCAN_label >= 0`, 즉 고립격자를 제외하고 실제 DBSCAN 취약권역에 포함된 격자만 집계한다.
- 대상 지수: 종합지수, 시설접근성지수.


In [ ]:
from pathlib import Path
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

BASE_PATH = Path.cwd().resolve()
if BASE_PATH.name == "dashboard":
    PROJECT_PATH = BASE_PATH.parents[1]
elif BASE_PATH.name == "notebooks":
    PROJECT_PATH = BASE_PATH.parent
elif (BASE_PATH / "notebooks").exists():
    PROJECT_PATH = BASE_PATH
else:
    PROJECT_PATH = Path(r"C:\project\oracle_mnc_project")

DBSCAN_GRID_PATH = PROJECT_PATH / "notebooks" / "dashboard" / "OUTPUT" / "vulnerability_index" / "DBSCAN_취약격자.csv"
dbscan_grid = pd.read_csv(DBSCAN_GRID_PATH, encoding="utf-8-sig")

included_grid = dbscan_grid[dbscan_grid["DBSCAN_label"].ge(0)].copy()
index_label = {
    "종합취약": "종합지수",
    "시설접근성취약": "시설접근성지수",
}
included_grid = included_grid[included_grid["권역유형"].isin(index_label.keys())].copy()
included_grid["지수"] = included_grid["권역유형"].map(index_label)

print("취약권역 포함 격자 수:", len(included_grid))
print(included_grid["지수"].value_counts())


## 1. 구별 취약권역 포함 격자 수


In [ ]:
gu_summary_all = (
    included_grid
    .groupby(["지수", "시군구"], as_index=False)
    .agg(
        취약권역포함격자수=("GRID_CD", "count"),
        취약권역수=("취약권역_ID", "nunique"),
        평균취약점수=("기준취약점수", "mean"),
        문화누리대상자수=("문화누리대상자_추정인구수", "sum"),
    )
)
gu_summary_all["평균취약점수"] = gu_summary_all["평균취약점수"].round(2)

gu_summary_종합지수 = gu_summary_all[gu_summary_all["지수"].eq("종합지수")].sort_values("취약권역포함격자수", ascending=False).reset_index(drop=True)
gu_summary_시설접근성지수 = gu_summary_all[gu_summary_all["지수"].eq("시설접근성지수")].sort_values("취약권역포함격자수", ascending=False).reset_index(drop=True)

print("종합지수 - 구별")
display(gu_summary_종합지수)
print("시설접근성지수 - 구별")
display(gu_summary_시설접근성지수)


## 2. 행정동별 취약권역 포함 격자 수


In [ ]:
dong_summary_all = (
    included_grid
    .groupby(["지수", "시군구", "행정동"], as_index=False)
    .agg(
        취약권역포함격자수=("GRID_CD", "count"),
        취약권역수=("취약권역_ID", "nunique"),
        평균취약점수=("기준취약점수", "mean"),
        문화누리대상자수=("문화누리대상자_추정인구수", "sum"),
    )
)
dong_summary_all["평균취약점수"] = dong_summary_all["평균취약점수"].round(2)

dong_summary_종합지수 = dong_summary_all[dong_summary_all["지수"].eq("종합지수")].sort_values("취약권역포함격자수", ascending=False).reset_index(drop=True)
dong_summary_시설접근성지수 = dong_summary_all[dong_summary_all["지수"].eq("시설접근성지수")].sort_values("취약권역포함격자수", ascending=False).reset_index(drop=True)

print("종합지수 - 행정동별")
display(dong_summary_종합지수)
print("시설접근성지수 - 행정동별")
display(dong_summary_시설접근성지수)
